# **Trim videos to discard unwanted spaces before and after the videos.**

In [2]:
!pip install numpy opencv-python matplotlib mediapipe pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.15.0 requires protobuf<4.21,>=3.20.3; python_version < "3.11", but you have protobuf 4.25.5 which is incompatible.


In [6]:
from google.colab import drive

# Check if the directory exists and is empty
import os
if os.path.exists('/content/drive') and os.listdir('/content/drive'):
  # If it's not empty, clear the directory
  !rm -rf /content/drive/*

# Now mount the drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
import os
# Usage example
''' Training data
csv_file = "/content/drive/MyDrive/videos_50/video_files.csv"  # Input CSV file with a column named 'file_name'
input_dir = "/content/drive/MyDrive/videos_50"  # Directory where input videos are located
output_dir = "/content/drive/MyDrive/videos_50_trimmed"  # Directory where trimmed videos will be saved
output_csv_file = output_dir + "/trimmed_videos_summary.csv" # Output summary CSV file
'''
#''' Validation data
csv_file = "/content/drive/MyDrive/ASL_Citizen_10_Test/video_files.csv"  # Input CSV file with a column named 'file_name'
input_dir = "/content/drive/MyDrive/ASL_Citizen_10_Test"  # Directory where input videos are located
output_dir = "/content/drive/MyDrive/ASL_Citizen_10_Test_trimmed"  # Directory where trimmed videos will be saved
output_csv_file = output_dir + "/trimmed_videos_summary.csv" # Output summary CSV file

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)


In [8]:
import cv2
import mediapipe as mp
import pandas as pd

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)

# Function to trim a video based on hand detection
def trim_video(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_path}")
        return None

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Variables to track trimming points
    start_trim_frame = None
    last_trim_frame = None
    frame_index = 0
    is_writing = False

    # Create VideoWriter object for the trimmed output file
    out = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)
    )
    trimmed_frame_count = 0

    # Step 1 and 2: Detect hand landmarks and trim in a single pass
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # Convert frame to RGB for MediaPipe processing
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        # Check if any hand landmarks are detected
        if results.multi_hand_landmarks:
            if start_trim_frame is None:
                start_trim_frame = frame_index  # Mark the first frame with hand detected
                is_writing = True  # Start writing to the output file
                print(f"Hand detected at frame {frame_index}. Starting trim...")

            last_trim_frame = frame_index  # Update last frame with hand detected
        else:
            # If hand landmarks are not visible and we are currently writing frames, stop the trim
            if is_writing:
                is_writing = False
                print(f"Hand lost at frame {frame_index}. Ending trim...")

        # Write frames to the output video only when in the trimming range
        if is_writing or (start_trim_frame is not None and frame_index <= last_trim_frame):
            out.write(frame)
            trimmed_frame_count += 1

        frame_index += 1

    cap.release()
    out.release()

    # If no hands were detected throughout the video, return None
    if start_trim_frame is None or last_trim_frame is None:
        print(f"Warning: No hand landmarks detected in {input_path}. Skipping trim.")
        os.remove(output_path)  # Remove the empty file
        return None

    return frame_count, trimmed_frame_count


# Main function to process videos from CSV file
def process_videos(csv_file, input_dir, output_dir, output_csv_file):
    # Read the CSV file containing video file names (without path)
    video_data = pd.read_csv(csv_file)
    summary_data = []

    # Process each video in the CSV
    for index, row in video_data.iterrows():
        file_name = row['file_name']

        # Full paths for input and output files
        input_file = os.path.join(input_dir, file_name)
        file_base, file_ext = os.path.splitext(file_name)
        output_file = os.path.join(output_dir, f"{file_base}_trim{file_ext}")

        print(f"Processing {input_file}...")

        # Trim the video and get frame count information
        result = trim_video(input_file, output_file)

        # If trimming was successful, record the information
        if result is not None:
            frame_count, trimmed_frame_count = result
            summary_data.append({
                "input_file": file_name,  # Use only the file name, not the full path
                "output_file": f"{file_base}_trim{file_ext}",  # Use only the file name, not the full path
                "frame_count_input": frame_count,
                "frame_count_output": trimmed_frame_count
            })
            print(f"Trimmed video saved to {output_file}. Input frames: {frame_count}, Output frames: {trimmed_frame_count}")
        else:
            print(f"Skipping {input_file} due to no hand landmarks detected.")

    # Write the summary data to a CSV file (without full paths)
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(output_csv_file, index=False)
    print(f"Summary of trimmed videos saved to {output_csv_file}")



# Run the video trimming process
process_videos(csv_file, input_dir, output_dir, output_csv_file)


Processing /content/drive/MyDrive/ASL_Citizen_10_Test/IMG_8909_Hello.mp4...


/usr/local/lib/python3.10/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Hand detected at frame 17. Starting trim...
Hand lost at frame 39. Ending trim...
Trimmed video saved to /content/drive/MyDrive/ASL_Citizen_10_Test_trimmed/IMG_8909_Hello_trim.mp4. Input frames: 100, Output frames: 45
Processing /content/drive/MyDrive/ASL_Citizen_10_Test/IMG_8910_Yes.mp4...
Hand detected at frame 20. Starting trim...
Hand lost at frame 80. Ending trim...
Trimmed video saved to /content/drive/MyDrive/ASL_Citizen_10_Test_trimmed/IMG_8910_Yes_trim.mp4. Input frames: 105, Output frames: 60
Processing /content/drive/MyDrive/ASL_Citizen_10_Test/IMG_8911_No.mp4...
Hand detected at frame 17. Starting trim...
Hand lost at frame 86. Ending trim...
Trimmed video saved to /content/drive/MyDrive/ASL_Citizen_10_Test_trimmed/IMG_8911_No_trim.mp4. Input frames: 107, Output frames: 69
Processing /content/drive/MyDrive/ASL_Citizen_10_Test/IMG_8912_Please.mp4...
Hand detected at frame 22. Starting trim...
Hand lost at frame 101. Ending trim...
Trimmed video saved to /content/drive/MyDriv